# 🧩 Mini-Lab: Conversation Memory

**Module 7: AI Agents** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** why agents need memory and the difference between short-term and long-term memory
2. **Implement** short-term memory by maintaining conversation history within a session
3. **Implement** long-term memory by persisting key facts to a JSON file across sessions
4. **Compare** agents with and without memory to see how memory affects conversation quality

## Target Concepts

| Concept | Description |
|---------|-------------|
| Agent Memory | The ability of an agent to retain and recall information, enabling contextual and personalized responses |
| Short-Term Memory | Session-scoped memory stored in the `messages` list — the agent remembers what was said earlier in the current conversation |
| Long-Term Memory | Persistent memory stored externally (e.g., file or database) — the agent remembers facts across separate conversations |

## Setup

In [2]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## Why Do Agents Need Memory?

In `mini-agent-loop` and `mini-react`, our agent loops used a `messages` list that grew with each iteration **within a single request**. But once the function returned, all that context was gone.

Consider a simple conversation:

```
User: My name is Alice.
Agent: Nice to meet you, Alice!

User: What's my name?
Agent: I don't know your name.  ← No memory!
```

Without memory, every message is processed in isolation. The agent has **amnesia**.

There are two kinds of memory an agent needs:

| Memory Type | Scope | Stored In | Analogy |
|-------------|-------|-----------|----------|
| **Short-Term** | Current conversation session | `messages` list in Python | Your working memory during a phone call |
| **Long-Term** | Across all sessions | External storage (file, DB) | Your notebook where you write down important facts |

## Part 1: No Memory (The Problem)

Let's first see what happens when an agent has **no memory** — each message is sent independently.

In [3]:
def chat_no_memory(user_message):
    """Send a single message with no conversation history."""
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[{"role": "user", "content": user_message}]
    )
    return response.choices[0].message.content

print("No-memory chat function defined.")

No-memory chat function defined.


In [4]:
md("**User:** My name is Alice and I'm a data scientist.")
reply = chat_no_memory("My name is Alice and I'm a data scientist.")
md(f"**Agent:** {reply}")

**User:** My name is Alice and I'm a data scientist.

**Agent:** Hi Alice! It's great to meet you. As a data scientist, you must have a lot of interesting insights and projects. How can I assist you today?

In [5]:
md("**User:** What's my name and what do I do?")
reply = chat_no_memory("What's my name and what do I do?")
md(f"**Agent:** {reply}")

**User:** What's my name and what do I do?

**Agent:** I'm sorry, but I don't have information about your name or what you do. If you'd like to share more details, I'd be happy to assist!

The agent has no idea who we are — the second call has zero context from the first. Each API call is **stateless**. Memory is something **we** must build.

## Part 2: Short-Term Memory

Short-term memory is the simplest form — we keep the full `messages` list and pass it to every API call. The LLM sees the entire conversation history and can refer back to anything said earlier.

This is **session-scoped**: it lives as long as our Python variable exists. When the session ends (notebook restarts, function returns), the memory is gone.

In [6]:
class ShortTermMemoryAgent:
    """An agent with short-term (session) memory."""
    
    def __init__(self, system_prompt="You are a helpful assistant."):
        self.messages = [{"role": "system", "content": system_prompt}]
    
    def chat(self, user_message):
        """Send a message and get a reply, maintaining conversation history."""
        self.messages.append({"role": "user", "content": user_message})
        
        response = client.chat.completions.create(
            model="gpt-4.1-nano",
            messages=self.messages
        )
        
        assistant_reply = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_reply})
        
        return assistant_reply
    
    def show_history(self):
        """Display the current conversation history."""
        for msg in self.messages:
            role = msg["role"].upper()
            md(f"**{role}:** {msg['content']}")

print("ShortTermMemoryAgent defined.")

ShortTermMemoryAgent defined.


In [7]:
agent = ShortTermMemoryAgent()

md("**User:** My name is Alice and I'm a data scientist.")
reply = agent.chat("My name is Alice and I'm a data scientist.")
md(f"**Agent:** {reply}")

**User:** My name is Alice and I'm a data scientist.

**Agent:** Hello Alice! It's great to meet a data scientist. How can I assist you today?

In [8]:
md("**User:** What's my name and what do I do?")
reply = agent.chat("What's my name and what do I do?")
md(f"**Agent:** {reply}")

**User:** What's my name and what do I do?

**Agent:** Your name is Alice, and you are a data scientist.

In [9]:
md("**User:** Recommend me a Python library related to my field.")
reply = agent.chat("Recommend me a Python library related to my field.")
md(f"**Agent:** {reply}")

**User:** Recommend me a Python library related to my field.

**Agent:** Certainly, Alice! A highly recommended Python library for data scientists is **pandas**. It provides powerful data structures and data analysis tools that make data manipulation and analysis much easier.

Other good options include:
- **NumPy**: For numerical computing and handling arrays.
- **scikit-learn**: For machine learning tasks.
- **Matplotlib / Seaborn**: For data visualization.
- **TensorFlow / PyTorch**: For deep learning.

If you'd like a recommendation tailored to a specific aspect of your work, please let me know!

The agent now remembers everything from the session. It knows Alice is a data scientist and uses that context for the recommendation.

Let's peek at the message history the LLM sees:

In [10]:
print(f"Messages in history: {len(agent.messages)}\n")
for i, msg in enumerate(agent.messages):
    role = msg['role']
    content = msg['content'][:80] + "..." if len(msg['content']) > 80 else msg['content']
    print(f"  [{i}] {role}: {content}")

Messages in history: 7

  [0] system: You are a helpful assistant.
  [1] user: My name is Alice and I'm a data scientist.
  [2] assistant: Hello Alice! It's great to meet a data scientist. How can I assist you today?
  [3] user: What's my name and what do I do?
  [4] assistant: Your name is Alice, and you are a data scientist.
  [5] user: Recommend me a Python library related to my field.
  [6] assistant: Certainly, Alice! A highly recommended Python library for data scientists is **p...


### The Limitation: New Session = Lost Memory

If we create a **new** agent instance (simulating a new session, a notebook restart, or a server reboot), all short-term memory is gone.

In [11]:
# Simulate a new session
new_agent = ShortTermMemoryAgent()

md("**User:** What's my name?")
reply = new_agent.chat("What's my name?")
md(f"**Agent:** {reply}")

**User:** What's my name?

**Agent:** I'm sorry, but I don't have access to your name. How can I assist you today?

Gone. The new agent has no knowledge of Alice. This is exactly the problem **long-term memory** solves.

## Part 3: Long-Term Memory

Long-term memory **persists across sessions** by storing important facts externally. We'll use a simple JSON file, but in production this could be a database, vector store, or any persistent storage.

Our approach:
1. Give the agent tools to **save** and **recall** facts
2. The agent decides what's worth remembering (via tool calls)
3. Facts are stored in a JSON file that survives restarts
4. At the start of each session, stored facts are loaded into the system prompt

In [12]:
MEMORY_FILE = "agent_memory.json"

def load_long_term_memory():
    """Load facts from the memory file."""
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r") as f:
            return json.load(f)
    return []

def save_long_term_memory(facts):
    """Save facts to the memory file."""
    with open(MEMORY_FILE, "w") as f:
        json.dump(facts, f, indent=2)

# Start fresh for this demo
save_long_term_memory([])
print(f"Long-term memory file: {MEMORY_FILE}")

Long-term memory file: agent_memory.json


In [13]:
def remember_fact(fact):
    """Store a fact in long-term memory."""
    facts = load_long_term_memory()
    facts.append(fact)
    save_long_term_memory(facts)
    return json.dumps({"status": "saved", "fact": fact})

def recall_facts():
    """Retrieve all stored facts from long-term memory."""
    facts = load_long_term_memory()
    return json.dumps({"facts": facts})


memory_tools = [
    {
        "type": "function",
        "function": {
            "name": "remember_fact",
            "description": "Save an important fact about the user to long-term memory. Use this when the user shares personal info, preferences, or anything worth remembering.",
            "parameters": {
                "type": "object",
                "properties": {
                    "fact": {
                        "type": "string",
                        "description": "The fact to remember, e.g. 'User's name is Alice'"
                    }
                },
                "required": ["fact"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "recall_facts",
            "description": "Retrieve all previously saved facts about the user from long-term memory.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]

available_functions = {
    "remember_fact": remember_fact,
    "recall_facts": recall_facts
}

print("Memory tools defined: remember_fact, recall_facts")

Memory tools defined: remember_fact, recall_facts


In [14]:
class LongTermMemoryAgent:
    """An agent with both short-term and long-term memory."""
    
    def __init__(self):
        # Load any existing long-term memories into the system prompt
        facts = load_long_term_memory()
        
        system = (
            "You are a helpful assistant with memory capabilities.\n"
            "When the user shares important personal information (name, job, "
            "preferences, etc.), use the remember_fact tool to save it.\n"
            "When the user asks about something you might have saved before, "
            "use recall_facts to check your memory.\n"
        )
        
        if facts:
            system += f"\nYou already know these facts from previous sessions:\n"
            for fact in facts:
                system += f"- {fact}\n"
        
        self.messages = [{"role": "system", "content": system}]
    
    def chat(self, user_message, max_iterations=5):
        """Chat with short-term history + long-term memory tools."""
        self.messages.append({"role": "user", "content": user_message})
        
        for _ in range(max_iterations):
            response = client.chat.completions.create(
                model="gpt-4.1-nano",
                messages=self.messages,
                tools=memory_tools
            )
            
            assistant_msg = response.choices[0].message
            finish_reason = response.choices[0].finish_reason
            
            if finish_reason == "stop":
                self.messages.append({"role": "assistant", "content": assistant_msg.content})
                return assistant_msg.content
            
            # Handle tool calls
            self.messages.append(assistant_msg)
            for tc in assistant_msg.tool_calls:
                args = json.loads(tc.function.arguments)
                fn = available_functions[tc.function.name]
                result = fn(**args) if args else fn()
                
                print(f"  🧠 {tc.function.name}({args}) → {result}")
                
                self.messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result
                })
        
        return "Max iterations reached."

print("LongTermMemoryAgent defined.")

LongTermMemoryAgent defined.


### Session 1: Introduce Ourselves

Let's tell the agent some facts. It should save them to long-term memory.

In [15]:
agent1 = LongTermMemoryAgent()

md("**User:** Hi! My name is Alice. I'm a data scientist and I love hiking.")
reply = agent1.chat("Hi! My name is Alice. I'm a data scientist and I love hiking.")
md(f"**Agent:** {reply}")

**User:** Hi! My name is Alice. I'm a data scientist and I love hiking.

  🧠 remember_fact({'fact': "User's name is Alice"}) → {"status": "saved", "fact": "User's name is Alice"}
  🧠 remember_fact({'fact': "User's profession is data scientist"}) → {"status": "saved", "fact": "User's profession is data scientist"}
  🧠 remember_fact({'fact': 'User loves hiking'}) → {"status": "saved", "fact": "User loves hiking"}


**Agent:** Nice to meet you, Alice! I've stored your name, profession, and your love for hiking. How can I assist you today?

In [16]:
md("**User:** My favorite programming language is Python.")
reply = agent1.chat("My favorite programming language is Python.")
md(f"**Agent:** {reply}")

**User:** My favorite programming language is Python.

  🧠 remember_fact({'fact': "User's favorite programming language is Python"}) → {"status": "saved", "fact": "User's favorite programming language is Python"}


**Agent:** I've noted that Python is your favorite programming language. How else can I help you today?

Let's verify the facts were saved to the file:

In [17]:
stored_facts = load_long_term_memory()
md("### 📁 Stored Long-Term Memory\n")
for i, fact in enumerate(stored_facts, 1):
    md(f"{i}. {fact}")

### 📁 Stored Long-Term Memory


1. User's name is Alice

2. User's profession is data scientist

3. User loves hiking

4. User's favorite programming language is Python

### Session 2: A Brand New Agent Instance

Now we simulate a **new session** — a completely new agent instance. This is like restarting the notebook or starting a new chat. The short-term memory (messages list) is empty, but the long-term memory file still exists.

In [18]:
# New session — fresh agent instance
agent2 = LongTermMemoryAgent()

md("**User:** Do you remember who I am?")
reply = agent2.chat("Do you remember who I am?")
md(f"**Agent:** {reply}")

**User:** Do you remember who I am?

  🧠 recall_facts({}) → {"facts": ["User's name is Alice", "User's profession is data scientist", "User loves hiking", "User's favorite programming language is Python"]}


**Agent:** Yes, I remember. You are Alice, a data scientist who loves hiking, and your favorite programming language is Python.

In [19]:
md("**User:** Based on what you know about me, suggest a weekend activity.")
reply = agent2.chat("Based on what you know about me, suggest a weekend activity.")
md(f"**Agent:** {reply}")

**User:** Based on what you know about me, suggest a weekend activity.

**Agent:** Since you love hiking, a great weekend activity would be planning a hiking trip to a scenic trail or a national park nearby. It would allow you to enjoy nature, get some exercise, and unwind. If you'd like, I can help you find specific trails or parks in your area.

The agent remembers Alice across sessions! It loaded the stored facts into the system prompt at initialization, so it knew the user's name, profession, and hobbies — even though this is a completely new instance.

This is the power of **long-term memory**.

## Comparing All Three Approaches

| Approach | Same Session? | New Session? | How It Works |
|----------|:---:|:---:|----|
| **No Memory** | ❌ | ❌ | Each message is independent — no `messages` history |
| **Short-Term Only** | ✅ | ❌ | Full `messages` list passed to each call — lost on restart |
| **Short-Term + Long-Term** | ✅ | ✅ | `messages` list for session + persistent file for key facts |

## Cleanup

Remove the memory file created during this lab.

In [20]:
if os.path.exists(MEMORY_FILE):
    os.remove(MEMORY_FILE)
    print(f"Cleaned up {MEMORY_FILE}")

Cleaned up agent_memory.json


## 🎯 Summary

### Key Takeaways

1. **Agent memory** — LLM APIs are stateless; memory is something we build by managing the `messages` list and external storage
2. **Short-term memory** — keeping the full conversation history in the `messages` list gives the agent context within a session, but is lost when the session ends
3. **Long-term memory** — persisting important facts to external storage (file, database) allows the agent to remember information across separate sessions
4. **Tools for memory** — giving the agent `remember_fact` and `recall_facts` tools lets the LLM decide what's worth saving, rather than storing everything blindly